<a href="https://colab.research.google.com/github/KBCoronado/MachineLearning/blob/main/Practica_5_Random_Forest_Prestamos_Lending_Club.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve


In [8]:
prestamos_df = pd.read_csv('lending_club_2007_2011_6_states (2).csv')
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [9]:
prestamos_df["grade_code"] = prestamos_df["grade"].astype("category").cat.codes
prestamos_df["purpose_code"] = prestamos_df["purpose"].astype("category").cat.codes
prestamos_df["addr_state_code"] = prestamos_df["addr_state"].astype("category").cat.codes
prestamos_df["home_ownership_code"] = prestamos_df["home_ownership"].astype("category").cat.codes

prestamos_df["repaid"] = prestamos_df["loan_status"].apply(
    lambda x: 1 if x == "Fully Paid" else 0
)

In [10]:
X = prestamos_df[['funded_amnt', 'int_rate', 'grade_code',
                  'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc',
                  'dti', 'revol_util',
                  'pub_rec_bankruptcies']].fillna(0)
y = prestamos_df["repaid"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.4, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

In [12]:
rfc1 = RandomForestClassifier(random_state=23)
rfc1 = rfc1.fit(X_train, y_train)
print("Precision del clasificador en fase de entrenamiento", rfc1.score(X_train, y_train))

Precision del clasificador en fase de entrenamiento 1.0


In [13]:
y_pred = rfc1.predict(X_test)
print("\nReporte del clasificador Random Forest sin balanceo de clases : \n",
      classification_report(y_test, y_pred, target_names=["No Pagado","Pagado"]))
print(f'\nMatriz de Confusion Random Forest sin balanceo de clases:\n', confusion_matrix(y_test, y_pred ))


Reporte del clasificador Random Forest sin balanceo de clases : 
               precision    recall  f1-score   support

   No Pagado       0.38      0.03      0.06      1177
      Pagado       0.86      0.99      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.62      0.51      0.49      7964
weighted avg       0.79      0.85      0.79      7964


Matriz de Confusion Random Forest sin balanceo de clases:
 [[  36 1141]
 [  58 6729]]


In [14]:
rfc2 = RandomForestClassifier(class_weight='balanced', random_state=23)
rfc2 = rfc2.fit(X_train, y_train)
y_pred = rfc2.predict(X_test)
print("Reporte del clasificador con balanceo de clases:\n", classification_report(y_test, y_pred,
                                                            target_names=["No Pagado","Pagado"] ))
print('Matriz de Confusion con balanceo de clases \n', confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases:
               precision    recall  f1-score   support

   No Pagado       0.36      0.02      0.03      1177
      Pagado       0.85      0.99      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.61      0.51      0.47      7964
weighted avg       0.78      0.85      0.79      7964

Matriz de Confusion con balanceo de clases 
 [[  19 1158]
 [  34 6753]]


In [15]:
rfc3 = RandomForestClassifier(n_estimators=200, max_depth=7, class_weight='balanced', random_state=23)
rfc3 = rfc3.fit(X_train, y_train)
y_pred = rfc3.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: \n",
      classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7\n' ,
      confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: 
               precision    recall  f1-score   support

   No Pagado       0.23      0.57      0.33      1177
      Pagado       0.90      0.66      0.76      6787

    accuracy                           0.65      7964
   macro avg       0.56      0.62      0.54      7964
weighted avg       0.80      0.65      0.70      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7
 [[ 674  503]
 [2295 4492]]


In [16]:
rfc4 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=23)
rfc4 = rfc4.fit(X_train, y_train)
y_pred = rfc4.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: \n",
      classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10\n' ,
      confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: 
               precision    recall  f1-score   support

   No Pagado       0.25      0.38      0.30      1177
      Pagado       0.88      0.81      0.84      6787

    accuracy                           0.74      7964
   macro avg       0.57      0.59      0.57      7964
weighted avg       0.79      0.74      0.76      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10
 [[ 444  733]
 [1319 5468]]


In [17]:
import plotly.graph_objects as go
modelos = [
    "RF sin balanceo",
    "RF balanceado",
    "RF bal. n=200, depth=7",
    "RF bal. n=200, depth=10"
]
accuracy = [0.85,0.85,0.65,0.73]
recall_no_pagado = [0.03,0.01,0.58,0.40]
recall_pagado = [0.99,1.00,0.66,0.78]
f1_macro = [0.49,0.47,0.54,0.57]
metricas = ["Accuracy","Recall No Pagado","Recall Pagado","F1 Macro"]
fig = go.Figure()
for i, modelo in enumerate (modelos):
  fig.add_trace(go.Scatterpolar(
      r=[accuracy[i], recall_no_pagado[i], recall_pagado[i], f1_macro[i], accuracy[i]],
      theta=metricas + [metricas[0]],
      fill='toself',
      name=modelo
  ))
fig.update_layout(
    title="Comparación de modelos Random Forest (RadarPlot)",
    polar=dict(
        radialaxis=dict(visible=
True, range=[0,1], tickvals=[0.2,0.4,0.6,0.8,1.0]),
        ),
    showlegend=True
)
fig.update_layout(width=800, height=600)
fig.show()


In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=1,
    verbose=2,
)
grid_search.fit(X_train, y_train)
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\n Mejor puntaje promedio (validación cruzada):")
print(grid_search.best_score_)
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print("\n Reporte de Clasificación (modelo óptimo):")
print(classification_report(y_test, y_pred_best, target_names=["No Pagado","Pagado"]))
print(confusion_matrix(y_test, y_pred_best))


Fitting 3 folds for each of 162 candidates, totalling 486 fits
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   1.4s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   1.4s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   1.4s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time=   2.7s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300; tota

In [19]:
scorer = make_scorer(recall_score, pos_label=0)
grid_search = GridSearchCV (
    RandomForestClassifier(class_weight='balanced', random_state=23),
    param_grid=param_grid,
    scoring=scorer,# << prioriza recall de la clase No Pagado
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [20]:
from sklearn.metrics import make_scorer, fbeta_score
scorer_f2 = make_scorer(fbeta_score, beta=2, pos_label=0)
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=23),
    param_grid=param_grid,
    scoring=scorer_f2,
    cv=5,
    n_jobs=-1,
    verbose=2
  )


In [23]:
from sklearn.metrics import f1_score
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=23),
    param_grid=param_grid,
    scoring={
        'recall_no_pagado': make_scorer(recall_score, pos_label=0),
        'f1_no_pagado': make_scorer(f1_score, pos_label=0),
        'accuracy':'accuracy'
},
refit='recall_no_pagado',# <--- elige el modelo que maximice recall de No Pagado
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [24]:
# Optimización de Random Forest priorizando detección de impagos

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, classification_report, confusion_matrix

# Definición del modelo base con balanceo de clases
modelo_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=23
)

# Definición de hiperparámetros a evaluar
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Definición del criterio de evaluación
# Se usará el recall de la clase "No Pagado" (pos_label=0)
scorer = make_scorer(recall_score, pos_label=0)

# Configuración del GridSearchCV
grid_search = GridSearchCV(
    estimator=modelo_base,
    param_grid=param_grid,
    scoring=scorer,  # prioriza recall de la clase No Pagado
    cv=5,            # validación cruzada con 5 particiones
    n_jobs=-1,       # usa todos los núcleos disponibles
    verbose=2
)

# Entrenamiento del modelo
grid_search.fit(X_train, y_train)

print("\nBúsqueda finalizada.")

print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")

# Evaluación del mejor modelo
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("\nReporte de Clasificación (modelo optimizado para detectar impagos):\n")

print(classification_report(y_test, y_pred))

print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 162 candidates, totalling 810 fits

Búsqueda finalizada.
Mejores hiperparámetros encontrados:
{'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300}

Reporte de Clasificación (modelo optimizado para detectar impagos):

              precision    recall  f1-score   support

           0       0.22      0.65      0.33      1177
           1       0.91      0.60      0.73      6787

    accuracy                           0.61      7964
   macro avg       0.57      0.63      0.53      7964
weighted avg       0.81      0.61      0.67      7964

Matriz de Confusión:
[[ 770  407]
 [2685 4102]]
